# 00 — Archive ingestion

Extract every unprocessed ZIP safely, load its CSV/Parquet files into typed-neutral
Delta tables in `archived`, and record file/run/table metrics in `monitoring`.

This notebook is idempotent at archive-file level. Set a file's `reload` flag to
`true` in `monitoring.cfg_archive_file_control` to process it again.

In [ ]:
# Parameters
BASE_DIR = "/lakehouse/default/Files"
SHORTCUT_PREFIX = "wmpp-production"
ARCHIVE_SUBFOLDER = "archive"
EXTRACT_ROOT = "/lakehouse/default/Files/archive_unzipped_v02"
ARCHIVE_SCHEMA = "archived"
TABLE_PREFIX = "archived_"
TEXT_QUALIFIER = '"'
FAIL_ON_FILE_ERROR = True

In [ ]:
import os, re, uuid, zipfile
from pathlib import Path
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.utcnow()

def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)

def safe_extract(zip_handle, destination):
    destination = Path(destination).resolve()
    for member in zip_handle.infolist():
        member_path = (destination / member.filename).resolve()
        if destination not in member_path.parents and member_path != destination:
            raise ValueError(f"Unsafe ZIP member path: {member.filename}")
    zip_handle.extractall(destination)

def clean_entity_name(file_name):
    stem = Path(file_name).stem.lower()
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", stem):
        return "audit"
    return re.sub(r"[^a-z0-9_]+", "_", stem).strip("_")

def upsert_archive_control(row, schema):
    source = spark.createDataFrame([row], schema)
    target = DeltaTable.forName(spark, "monitoring.cfg_archive_file_control")
    (target.alias("t").merge(source.alias("s"), "t.archive_path = s.archive_path")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT, rows_written BIGINT,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, null_primary_key_count BIGINT,
  recorded_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ARCHIVE_SCHEMA}")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_file_control (
  archive_path STRING, archive_name STRING, reload BOOLEAN, export_date DATE,
  extract_status STRING, extracted_file_count INT, loaded_table_count INT,
  rows_written BIGINT, first_processed_at TIMESTAMP, last_processed_at TIMESTAMP,
  run_id STRING, error_message STRING
) USING DELTA
""")
spark.createDataFrame([(RUN_ID, "00_archive_load", "BRONZE", "ARCHIVE", RUN_STARTED_AT, None,
                        "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string") \
    .write.format("delta").mode("append").saveAsTable("monitoring.cfg_pipeline_run")

In [ ]:
shortcut_candidates = sorted(
    name for name in os.listdir(BASE_DIR)
    if name.startswith(SHORTCUT_PREFIX)
)
if not shortcut_candidates:
    raise FileNotFoundError(f"No shortcut under {BASE_DIR} starts with {SHORTCUT_PREFIX!r}")

archive_dir = os.path.join(BASE_DIR, shortcut_candidates[0], ARCHIVE_SUBFOLDER)
zip_names = sorted(name for name in os.listdir(archive_dir) if name.lower().endswith(".zip"))

processed = spark.table("monitoring.cfg_archive_file_control") \
    .where("extract_status = 'SUCCESS' AND coalesce(reload, false) = false") \
    .select("archive_path").distinct()
processed_paths = {row.archive_path for row in processed.collect()}

control_schema = "archive_path string,archive_name string,reload boolean,export_date date,extract_status string,extracted_file_count int,loaded_table_count int,rows_written long,first_processed_at timestamp,last_processed_at timestamp,run_id string,error_message string"
metric_schema = "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,null_primary_key_count long,recorded_at timestamp"

tables_ok = tables_failed = rows_read_total = rows_written_total = 0
errors = []

for zip_name in zip_names:
    zip_path = os.path.join(archive_dir, zip_name)
    relative_zip_path = os.path.relpath(zip_path, "/lakehouse/default")
    if relative_zip_path in processed_paths:
        print(f"SKIP already processed: {relative_zip_path}")
        continue

    export_text = Path(zip_name).stem
    export_date = datetime.strptime(export_text, "%Y-%m-%d").date() if re.fullmatch(r"\d{4}-\d{2}-\d{2}", export_text) else None
    destination = os.path.join(EXTRACT_ROOT, export_text)
    os.makedirs(destination, exist_ok=True)
    archive_rows = 0
    loaded_tables = set()
    extracted_files = 0

    try:
        with zipfile.ZipFile(zip_path, "r") as handle:
            safe_extract(handle, destination)
            extracted_files = len([m for m in handle.infolist() if not m.is_dir()])

        for root, _, files in os.walk(destination):
            for file_name in sorted(files):
                if not file_name.lower().endswith((".csv", ".parquet")):
                    continue
                file_path = os.path.join(root, file_name)
                relative_path = os.path.relpath(file_path, "/lakehouse/default")
                entity = clean_entity_name(file_name)
                target = f"{ARCHIVE_SCHEMA}.{TABLE_PREFIX}{entity}"

                # File-level idempotence: remove an earlier partial attempt before rewriting this file.
                escaped_relative_path = relative_path.replace("'", "''")
                if spark.catalog.tableExists(target) and "_source_file_path" in spark.table(target).columns:
                    spark.sql(f"DELETE FROM {target} WHERE _source_file_path = '{escaped_relative_path}'")

                if file_name.lower().endswith(".parquet"):
                    frame = spark.read.format("parquet").load(relative_path)
                else:
                    frame = (spark.read.format("csv").option("header", "true")
                        .option("inferSchema", "false").option("mode", "PERMISSIVE")
                        .option("quote", TEXT_QUALIFIER).option("escape", TEXT_QUALIFIER)
                        .option("multiLine", "true").load(relative_path))

                frame = (frame.withColumn("_source_file_path", F.lit(relative_path))
                    .withColumn("_source_archive_path", F.lit(relative_zip_path))
                    .withColumn("_export_date", F.lit(export_date).cast("date"))
                    .withColumn("_archive_run_id", F.lit(RUN_ID))
                    .withColumn("_archive_load_ts", F.current_timestamp()))
                count = frame.count()
                frame.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target)
                append_rows("monitoring.cfg_table_load_metric", [(RUN_ID, "BRONZE", "ARCHIVE", relative_path,
                    target, count, count, None, None, datetime.utcnow())], metric_schema)
                archive_rows += count
                loaded_tables.add(target)

        upsert_archive_control((relative_zip_path, zip_name, False, export_date,
            "SUCCESS", extracted_files, len(loaded_tables), archive_rows, datetime.utcnow(), datetime.utcnow(),
            RUN_ID, None), control_schema)
        tables_ok += len(loaded_tables)
        rows_read_total += archive_rows
        rows_written_total += archive_rows
    except Exception as exc:
        message = str(exc)[:2000]
        errors.append(f"{zip_name}: {message}")
        tables_failed += 1
        upsert_archive_control((relative_zip_path, zip_name, False, export_date,
            "FAILED", extracted_files, len(loaded_tables), archive_rows, datetime.utcnow(), datetime.utcnow(),
            RUN_ID, message), control_schema)
        if FAIL_ON_FILE_ERROR:
            break

final_status = "FAILED" if errors else "SUCCESS"
error_text = " | ".join(errors)[:4000] if errors else None
error_sql = "NULL" if error_text is None else "'" + error_text.replace("'", "''") + "'"
spark.sql(f"""
UPDATE monitoring.cfg_pipeline_run
SET ended_at=current_timestamp(), status='{final_status}', tables_succeeded={tables_ok},
    tables_failed={tables_failed}, rows_read={rows_read_total}, rows_written={rows_written_total},
    error_message={error_sql}
WHERE run_id='{RUN_ID}'
""")
if errors and FAIL_ON_FILE_ERROR:
    raise RuntimeError(error_text)
print(f"Archive run {RUN_ID}: {final_status}; {rows_written_total:,} rows written")